# نَبْض — إنتاج صوت المعلّمة B للأحياء

هذه تجربة مجانية مفتوحة المصدر لتوليد تسجيلات دروس **الأحياء** من تسجيل **B** الاصطناعي الذي رفعته أنت. يستخدم النموذج العراقي المتخصص **Habibi-TTS Specialized IRQ**.

- المصدر الصوتي هو ملفك الخاص `nabd_teacher_selected_B.mp3`، لا ترفعه إلى GitHub العام.
- **لا يُضمَن التطابق 100%** مع صوت المرجع أو دقة اللهجة قبل سماع التسجيلات الناتجة.
- النسخة المتخصصة العراقية تحت **Apache-2.0**؛ أما حقوق إعادة اشتقاق صوت NotebookLM فتحتاج مراجعة مستقلة.
- شغّل الدفتر في Google Colab بعد اختيار GPU (قد لا تتوفر GPU مجانية في حسابك).
- أدخل **التفريغ الحرفي الصحيح** للصوت المرجعي بعد سماعه؛ لا تستعمل نصاً تخمينياً.

المراجع: https://github.com/SWivid/Habibi-TTS


In [ ]:
# 1. تهيئة النموذج والتأكد من توفر GPU
import sys,subprocess,shutil
import torch
assert torch.cuda.is_available(), "فعّل GPU من إعدادات تشغيل Colab. بدونها التوليد غير متاح في هذا الدفتر."
print("GPU:",torch.cuda.get_device_name(0))
subprocess.run([sys.executable,"-m","pip","install","-q","habibi-tts"],check=True)
assert shutil.which("habibi-tts_infer-cli"),"تعذر تثبيت محرك Habibi-TTS"
print("✅ البيئة جاهزة؛ عند التوليد الأول ستنزل الأوزان من المصدر المفتوح")


In [ ]:
# 2. رفع مرجع B الخاص إلى Colab وتنزيل نصوص الدروس العامة
from google.colab import files
from pathlib import Path
import json,hashlib,requests
data=files.upload()
assert len(data)==1, "ارفع ملف MP3 المرجعي B فقط"
reference=Path(next(iter(data)))
assert reference.name=="nabd_teacher_selected_B.mp3","ارفع الملف المرجعي المرفق باسمه الأصلي"
sha=hashlib.sha256(reference.read_bytes()).hexdigest()
assert sha=="c2a2ab1834b9d3e2d205fd4a6d228ca8fac30a37c0212d9bcf6206aebdeff618", "هذا ليس نفس تسجيل B الذي اخترته"
url="https://raw.githubusercontent.com/alians200a-commits/Nbd/main/data/NABD_B_44_VOICE_JOBS.json"
response=requests.get(url,timeout=30);response.raise_for_status()
jobs_json=response.json()
assert jobs_json["format"]=="NABD_VOICE_PRODUCTION_JOBS" and jobs_json["speaker"]=="B" and len(jobs_json["jobs"])==44
jobs=jobs_json["jobs"]
print("✅ تسجيل B متطابق وعدد دروس الأحياء:",len(jobs))


In [ ]:
# 3. اسمع المرجع وصحح نصه الحرفي
from IPython.display import Audio,display
display(Audio(filename=str(reference)))
ref_text=input("ألصق تفريغ الكلام المسموع بصوت B حرفياً: ").strip()
assert len(ref_text)>=25,"يجب تقديم تفريغ صحيح للمرجع"
voice_ok=input("هل الصوت واضح لمتحدث واحد، والتفريغ مطابق، وراجعت حقوق إعادة الاستخدام؟ اكتب نعم: ").strip()
assert voice_ok=="نعم","لا نكمل من دون التحقق من الصوت والحقوق"
choice=input("كم درس نجرب أولاً؟ 3 للتجربة، 11 أو 44: ").strip() or "3"
assert choice in ("3","11","44")
selected=jobs[:int(choice)]
print("عدد التسجيلات المطلوب توليدها:",len(selected))


In [ ]:
# 4. التوليد الفعلي باللهجة العراقية المتخصصة (قد يحتاج تحميل نموذج كبير)
import subprocess
out=Path("/content/NABD_B_generated");out.mkdir(exist_ok=True)
done=[];failed=[]
for i,j in enumerate(selected,1):
  target=out/(j["id"]+".wav")
  cmd=["habibi-tts_infer-cli","--model","Specialized","--dialect","IRQ",
       "--ref_audio",str(reference),"--ref_text",ref_text,"--gen_text",j["script"],
       "--output_dir",str(out),"--output_file",target.name]
  print(f"توليد {i}/{len(selected)}: {j['title']}")
  res=subprocess.run(cmd,check=False)
  if res.returncode!=0 or not target.exists() or target.stat().st_size<1024:
    print("فشل توليد:",j["id"]);failed.append(j["id"]);continue
  mp3=out/(j["id"]+".mp3")
  res=subprocess.run(["ffmpeg","-y","-v","error","-i",str(target),"-ar","24000","-ac","1","-b:a","96k",str(mp3)],check=False)
  if res.returncode!=0 or not mp3.exists():failed.append(j["id"]);continue
  done.append(j)
print("✅ تسجيلات تجريبية ناجحة:",len(done),"فشل:",len(failed))
assert done,"فشل التوليد. راجع سجل الخطأ أعلاه."


In [ ]:
# 5. فحص جودة تقني للتسجيلات قبل اعتبارها مقاطع صالحة للمراجعة السمعية
# هذا الفحص يكشف التسجيلات التالفة/الصامتة/المبتورة فقط ولا يثبت تطابق الهوية الصوتية.
import numpy as np, soundfile as sf
good, rejected, diagnostics = [], [], []
for j in done:
    wav = out / (j["id"]+".wav")
    mp3 = out / (j["id"]+".mp3")
    if not wav.exists() or not mp3.exists():
        rejected.append(j["id"]);continue
    y, fs = sf.read(str(wav))
    if y.ndim > 1: y = y.mean(axis=1)
    length = len(y) / fs if fs else 0
    peak = float(np.max(np.abs(y))) if len(y) else 0
    rms = float(np.sqrt(np.mean(np.square(y)))) if len(y) else 0
    silent = float(np.mean(np.abs(y) < 0.002)) if len(y) else 1
    ok = 3 <= length <= 720 and 0.008 <= rms <= 0.65 and peak <= 1.01 and silent < 0.96
    diagnostics.append({"id":j["id"],"seconds":round(length,1),"rms":round(rms,4),
                        "peak":round(peak,4),"silence_ratio":round(silent,3),"passed":ok})
    (good if ok else rejected).append(j if ok else j["id"])
print("تم فحص:",len(diagnostics),"صالح للتدقيق السمعي:",len(good),"مرفوض تقنياً:",len(rejected))
for d in diagnostics[:10]:print(d)
assert good,"لم ينجح أي ملف في فحص الصوت التقني. راجع النموذج وإعدادات الاستدلال."
done = good



In [ ]:
# 5. استمع للتسجيلات الجديدة وقيّم تقاربها من صوت B يدوياً قبل التصدير
for j in done[:3]:
  print(j["title"],"— الفصل",j["chapter"])
  display(Audio(filename=str(out/(j["id"]+".mp3"))))
approval=input("هل راجعت النطق العراقي وتشابه صوت المعلّمة B في جميع التسجيلات المراد تصديرها؟ اكتب نعم: ").strip()
assert approval=="نعم","لا يجوز نشر تسجيلات لم تُراجع"


In [ ]:
# 6. تصدير حزمة جميع المقاطع التي نجحت — استوردها في نَبْض من قائمة صوت B
import base64,hashlib,json
pack={"format":"NABD_B_LESSON_AUDIO_PACK_V1","speaker":"B","version":1,
      "model":"Habibi-TTS Specialized IRQ","reviewedByUser":True,"clips":[]}
for j in done:
  binary=(out/(j["id"]+".mp3")).read_bytes()
  pack["clips"].append({"key":j["key"],"sha256":hashlib.sha256(binary).hexdigest(),
      "base64":base64.b64encode(binary).decode("ascii")})
path=Path("/content/NABD_B_AUDIO_LESSONS.json")
path.write_text(json.dumps(pack,ensure_ascii=False),encoding="utf-8")
files.download(str(path))
print("✅ حزمة تسجيلات B جاهزة:",len(pack["clips"]),"مقطعاً")
print("من تطبيق نَبْض: افتح مكتبة الدروس > أيقونة الصوت > استورد مرجع B أولاً ثم الحزمة.")
